# Siver Data Transformation

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [ ]:
df = spark.read.format("delta")\
    .option("header",True)\
    .option("inferSchema",True)\
    .load("abfss://bronze@netflixprojectdlrayyan.dfs.core.windows.net/netflix_titles")

#

In [ ]:
df.display()

In [ ]:
df = df.fillna({"duration_minutes": 0, "duration_seasons": 1})

In [ ]:
df.display()

In [ ]:
from pyspark.sql.functions import expr

df = df.withColumn("duration_minutes", expr("try_cast(duration_minutes as int)"))\
       .withColumn("duration_seasons", expr("try_cast(duration_seasons as int)"))

In [ ]:
df.printSchema()

In [ ]:
df.display()

In [ ]:
df = df.withColumn("Shorttitle", split(col("title"), ':')[0])
df.display()

In [ ]:
df = df.withColumn("rating",split(col("rating"), "-")[0])

In [ ]:
df.display()

### **Transforming the "Type" Col into When/Otherwise flagging to 0 and 1**

In [ ]:
df = df.withColumn("type_flag",when(col('type')=='Movie',1)\
                   .when(col('type')=='TV Show',2)\
                   .otherwise(0))
display(df)

In [ ]:
from pyspark.sql.window import Window

In [ ]:
df = df.withColumn("duration_ranking", dense_rank().over(Window.orderBy(col('duration_minutes').desc())))

In [ ]:
display(df)

In [ ]:
df.createOrReplaceTempView("temp_view") #THESE ARE TEMPORARY VIEWS WE CAN USE IN THIS NOTEBOOK

In [ ]:
# df.createOrReplaceGlobalTempView("global_view") THESE TEMPORARY VIEWS WE CAN USE IN OTHER NOTEBOOKS AS WELL

In [ ]:
df = spark.sql("""
       select * from temp_view
       
                """)

In [ ]:
display(df)

# **Writing the Data**

In [ ]:
df.write.format("delta")\
  .mode("overwrite")\
  .option("overwriteSchema", "true")\
  .option("path", "abfss://silver@netflixprojectdlrayyan.dfs.core.windows.net/netflix_titles")\
  .save()
